In [1]:
import numpy as np
import pandas as pd
import sympy as sp
from scipy.optimize import differential_evolution

pd.set_option('display.precision', 15)  # Increase decimal precision
pd.set_option('display.width', 150)     # Wider display
pd.set_option('display.max_columns', None)  # Show all column

# Phương pháp Lặp đơn cho Hệ phi tuyến

Điều kiện hội tụ: 

* $G(D) \subset D, \forall X \in D $

* $||J|| \leq q < 1$ --> đảm bảo nghiệm duy nhất ($||\Phi(X)-\Phi(Y)|| \leq q||X-Y||$)

Đánh giá sai số (hậu nghiệm) theo 1 trong 2 công thức sau:

* $||x_n - x^*|| \leq \dfrac{q^n}{1-q} ||x_1-x_0||$ $\quad$ (1)

* $||x_n - x^*|| \leq \dfrac{q}{1-q} ||x_n-x_{n-1}||$ $\quad$ (2)



# Thuật toán

#### Input:
- Hàm lặp $\Phi(X)=(\varphi_1, \varphi_2, \ldots, \varphi_n)$ ứng với phương trình lặp $X=\Phi(X)$ tương đương với phương trình $F(X)=0$
- Miền $D=[a_1,b_1] \times [a_2,b_2] \times ... \times [a_n,b_n]$
- Sai số $\varepsilon$
#### Output: nghiệm gần đúng X

#### Thuật toán:
- Bước 1: Kiểm tra $\Phi(X) \in D \,\, \forall X \in D$. Nếu không thoả mãn, chọn miền $D$ khác. Nếu thoả mãn $\Rightarrow$ Bước 2
- Bước 2: Lập ma trận $D_\varphi$ = $[a_{ij}]_{m \times n}$ với $a_{ij} = \dfrac{\partial\varphi_i}{\partial x_j}$
- Bước 3: Tính $q$ theo chuẩn hàng hoặc chuẩn cột:
  - Chuẩn hàng: $q = \max\limits_D ||D_\varphi||_{\infty} = \max\limits_{1 \leq i \leq m} \displaystyle \sum\limits_{j=1}^n |a_{ij}|$
  - Chuẩn cột: $q = \max\limits_D ||D_\varphi||_1 = \max\limits_{1 \leq j \leq n} \displaystyle \sum\limits_{i=1}^m |a_{ij}|$

  Nếu $q < 1 \Rightarrow $ $\Phi$ co trên $D \Rightarrow$ Thoả mãn điều kiện phương pháp lặp đơn
- Bước 4: Chọn điểm bắt đầu $X_0$, khởi tạo $k=0$. Lặp đến khi thoả mãn điều kiện dừng $||X_{k+1} - X_k|| \leq \dfrac{(1-q)\varepsilon}{q}$:
  - Tính $X_{k+1} = (x_{1,k+1};x_{2,k+1};x_{3,k+1}) = \Phi(X_k), \quad ||X_{k+1}-X_k||$
  - Nếu thoả mãn điều kiện dừng: dừng chương trình. Trái lại cập nhật $k=k+1$
- Bước 5: Sau khi thoả mãn điều kiện dừng với $n$ lần lặp, thu được nghiệm gần đúng là $X=\Phi(X_{n-1})$. Dừng chương trình

# Áp dụng

* Các chuẩn vecto: $||x||_{(1)} = \displaystyle \sum\limits_{i=1}^{k} |x_{i}|$, $\quad$ $||x||_{(\infty)} = \max\limits_{i} |x_{i}|$
* Các chuẩn ma trận (tính hệ số co):
    * Chuẩn max: $q = m * \max\limits_{i,j} |G_{ij}|$, $\quad$ (chuẩn vecto 1 và chuẩn vecto $\infty$)
    * Chuẩn cột: $q = ||G||_{(1)} = \max\limits_{1\leq j \leq n} \displaystyle \sum\limits_{i=1}^{m} |G_{ij}|$, $\quad$ (chuẩn vecto 1)
    * Chuẩn hàng: $q = ||G||_{(\infty)} = \max\limits_{1\leq i \leq m} \displaystyle \sum\limits_{j=1}^{n} |G_{ij}|$, $\quad$ (chuẩn vecto $\infty$)

## 1. Sai số tuyệt đối

In [2]:
def fixed_point_recursion_absolute (initial_values, q, eps, *funcs, norm):
    n = len(initial_values)
    values = np.array(initial_values)
    results = []
    i=0

    new_eps = eps * (1-q)/ q
    print(f"new_eps: {new_eps}")

    while True:
        new_values = np.array([funcs[j](*values) for j in range(n)])
        
        # Calculate the differences
        diffs = np.abs(new_values - values)
        if (norm == 1): total_diff = np.sum(diffs)
        elif (norm == -1): total_diff = np.max(diffs)
        
        # Append the iteration number, new values, differences, and total_diff to the results list
        results.append([i] + new_values.tolist() + diffs.tolist() + [total_diff])
        values = new_values
        
        
        # Check for convergence
        if total_diff < new_eps:
            break
        i += 1  # Increment the iteration counter
        
    columns = ['k'] + [f'x_({j+1},k+1)' for j in range(n)] + [f'diff_x({j+1},k+1)' for j in range(n)] + ['total_diff']
    df = pd.DataFrame(results, columns=columns)
    print(df.to_string(index=False))  # Print the DataFrame without the index
    
    return values

def check_GD(g_funcs, bounds):
    print("--- CHI TIẾT MIỀN GIÁ TRỊ G(D) ---")
    for i, g_func in enumerate(g_funcs):
        def func_wrapper(vars_array, f=g_func): return float(f(*vars_array))
        def neg_func_wrapper(vars_array, f=g_func): return -float(f(*vars_array))
        g_min = differential_evolution(func_wrapper, bounds).fun
        g_max = -differential_evolution(neg_func_wrapper, bounds).fun
        b_min, b_max = bounds[i]
        print(f"g{i+1}(D): [{g_min:.6f}, {g_max:.6f}] | Giới hạn D{i+1}: [{b_min}, {b_max}]")

def calc_norms(g_funcs, bounds, vars_symbols):
    G_expr = [func(*vars_symbols) for func in g_funcs]
    J = sp.Matrix(G_expr).jacobian(sp.Matrix(vars_symbols))
    J_func = sp.lambdify(vars_symbols, J, 'numpy')
    
    print("\n--- CHI TIẾT TÍNH CHUẨN HÀNG ---")
    row_maxes = []
    for i in range(J.shape[0]):
        def neg_row_i_norm(vars_array, row_idx=i): return -np.sum(np.abs(J_func(*vars_array)[row_idx, :]))
        q_i = -differential_evolution(neg_row_i_norm, bounds).fun
        row_maxes.append(q_i)
        print(f"Max Hàng {i+1} (q{i+1}): {q_i:.4f}")
    row_norm = max(row_maxes)
    print(f"=> Chuẩn hàng q = max({', '.join([f'{q:.4f}' for q in row_maxes])}) = {row_norm:.4f}")
    
    print("\n--- CHI TIẾT TÍNH CHUẨN CỘT ---")
    col_maxes = []
    for j in range(J.shape[1]):
        def neg_col_j_norm(vars_array, col_idx=j): return -np.sum(np.abs(J_func(*vars_array)[:, col_idx]))
        c_j = -differential_evolution(neg_col_j_norm, bounds).fun
        col_maxes.append(c_j)
        print(f"Max Cột {j+1} (c{j+1}): {c_j:.4f}")
    col_norm = max(col_maxes)
    print(f"=> Chuẩn cột q = max({', '.join([f'{c:.4f}' for c in col_maxes])}) = {col_norm:.4f}")
    return row_norm, col_norm

def run_iteration(g_funcs, vars_symbols, initial_values, q, eps, norm=-1):
    G_expr = [func(*vars_symbols) for func in g_funcs]
    G_numpy = sp.lambdify(vars_symbols, G_expr, 'numpy')
    g_funcs_np = [lambda *args, idx=i: G_numpy(*args)[idx] for i in range(len(vars_symbols))]
    
    solution = fixed_point_recursion_absolute(initial_values, q, eps, *g_funcs_np, norm=norm)
    print("Approximate solution:", solution)
    return solution

Bài tập VD trên lớp:

In [3]:
def g1(x, y, z):
    return 1/3 * (sp.cos(y*z) + 0.5)

def g2(x, y, z):
    return 1/9 * sp.sqrt(x**2 + sp.sin(z) + 1.06) - 0.1

def g3(x, y, z):
    return -1/20 * sp.exp(-x*y) - 9.1389/20

g_funcs = [g1, g2, g3]
bounds = [(-1.0, 1.0), (-1.0, 1.0), (-1.0, 1.0)]
check_GD(g_funcs, bounds)

--- CHI TIẾT MIỀN GIÁ TRỊ G(D) ---
g1(D): [0.346767, 0.500000] | Giới hạn D1: [-1.0, 1.0]
g2(D): [-0.048059, 0.089263] | Giới hạn D2: [-1.0, 1.0]
g3(D): [-0.592859, -0.475339] | Giới hạn D3: [-1.0, 1.0]


In [4]:
vars_symbols = sp.symbols('x y z')
row_norm, column_norm = calc_norms(g_funcs, bounds, vars_symbols)


--- CHI TIẾT TÍNH CHUẨN HÀNG ---
Max Hàng 1 (q1): 0.5610
Max Hàng 2 (q2): 0.1293
Max Hàng 3 (q3): 0.2718
=> Chuẩn hàng q = max(0.5610, 0.1293, 0.2718) = 0.5610

--- CHI TIẾT TÍNH CHUẨN CỘT ---
Max Cột 1 (c1): 0.2366
Max Cột 2 (c2): 0.4164
Max Cột 3 (c3): 0.3447
=> Chuẩn cột q = max(0.2366, 0.4164, 0.3447) = 0.4164


In [5]:
initial_values = [0, 0, 0]
eps = 1e-4
q = row_norm
solution = run_iteration(g_funcs, vars_symbols, initial_values, q, eps, norm=-1)

new_eps: 7.825926586671837e-05
 k         x_(1,k+1)         x_(2,k+1)          x_(3,k+1)     diff_x(1,k+1)     diff_x(2,k+1)     diff_x(3,k+1)        total_diff
 0 0.500000000000000 0.014395890455411 -0.506945000000000 0.500000000000000 0.014395890455411 0.506945000000000 0.506945000000000
 1 0.499991123421941 0.000890556858707 -0.506586394896849 0.000008876578059 0.013505333596704 0.000358605103151 0.013505333596704
 2 0.499999966078184 0.000909195229164 -0.506922741429698 0.000008842656243 0.000018638370457 0.000336346532849 0.000336346532849
 3 0.499999964596468 0.000891745362941 -0.506922275286504 0.000000001481716 0.000017449866224 0.000000466143193 0.000017449866224
Approximate solution: [ 0.49999996  0.00089175 -0.50692228]


Câu 20a:

In [6]:
def g1(x, y, z):
    return 1/3 * (sp.cos(y*z) + 0.5)

def g2(x, y, z):
    return 1/25 * sp.sqrt(x**2 + 0.3125) - 0.03

def g3(x, y, z):
    return -1/20 * sp.exp(-x*y) - (10*sp.pi-3)/60

g_funcs = [g1, g2, g3]
bounds = [(-1.0, 1.0), (-1.0, 1.0), (-1.0, 1.0)]
check_GD(g_funcs, bounds)

--- CHI TIẾT MIỀN GIÁ TRỊ G(D) ---
g1(D): [0.346767, 0.500000] | Giới hạn D1: [-1.0, 1.0]
g2(D): [-0.007639, 0.015826] | Giới hạn D2: [-1.0, 1.0]
g3(D): [-0.609513, -0.491993] | Giới hạn D3: [-1.0, 1.0]


In [7]:
vars_symbols = sp.symbols('x y z')
row_norm, column_norm = calc_norms(g_funcs, bounds, vars_symbols)


--- CHI TIẾT TÍNH CHUẨN HÀNG ---
Max Hàng 1 (q1): 0.5610
Max Hàng 2 (q2): 0.0349
Max Hàng 3 (q3): 0.2718
=> Chuẩn hàng q = max(0.5610, 0.0349, 0.2718) = 0.5610

--- CHI TIẾT TÍNH CHUẨN CỘT ---
Max Cột 1 (c1): 0.1708
Max Cột 2 (c2): 0.4164
Max Cột 3 (c3): 0.2805
=> Chuẩn cột q = max(0.1708, 0.4164, 0.2805) = 0.4164


In [8]:
initial_values = [0, 0, 0]
eps = 1e-6
q = column_norm
solution = run_iteration(g_funcs, vars_symbols, initial_values, q, eps, norm=1)

new_eps: 1.4015114938959177e-06
 k       x_(1,k+1)          x_(2,k+1)          x_(3,k+1)         diff_x(1,k+1)     diff_x(2,k+1)     diff_x(3,k+1)        total_diff
 0 0.5000000000000 -0.007639320225002 -0.523598775598299 5.000000000000000e-01 0.007639320225002 0.523598775598299 1.031238095823301
 1 0.4999973334186  0.000000000000000 -0.523790123813853 2.666581400245782e-06 0.007639320225002 0.000191348215554 0.007833335021956
 2 0.5000000000000 -0.000000071108732 -0.523598775598299 2.666581400245782e-06 0.000000071108732 0.000191348215554 0.000194085905686
 3 0.5000000000000  0.000000000000000 -0.523598777376017 2.220446049250313e-16 0.000000071108732 0.000000001777718 0.000000072886451
Approximate solution: [ 0.5         0.         -0.52359878]


Câu 20b:

In [9]:
def g1(x, y, z):
    return (13 - y**2 + 4*z) / 15

def g2(x, y, z):
    return (11 + z - x**2) / 10

def g3(x, y, z):
    return (22 + y**3) / 25

g_funcs = [g1, g2, g3]
bounds = [(0, 1.5), (0, 1.5), (0, 1.5)]
check_GD(g_funcs, bounds)

--- CHI TIẾT MIỀN GIÁ TRỊ G(D) ---
g1(D): [0.716667, 1.266667] | Giới hạn D1: [0, 1.5]
g2(D): [0.875000, 1.250000] | Giới hạn D2: [0, 1.5]
g3(D): [0.880000, 1.015000] | Giới hạn D3: [0, 1.5]


In [10]:
vars_symbols = sp.symbols('x y z')
row_norm, column_norm = calc_norms(g_funcs, bounds, vars_symbols)


--- CHI TIẾT TÍNH CHUẨN HÀNG ---
Max Hàng 1 (q1): 0.4667
Max Hàng 2 (q2): 0.4000
Max Hàng 3 (q3): 0.2700
=> Chuẩn hàng q = max(0.4667, 0.4000, 0.2700) = 0.4667

--- CHI TIẾT TÍNH CHUẨN CỘT ---
Max Cột 1 (c1): 0.3000
Max Cột 2 (c2): 0.4700
Max Cột 3 (c3): 0.3667
=> Chuẩn cột q = max(0.3000, 0.4700, 0.3667) = 0.4700


In [11]:
initial_values = [0, 0, 0]
eps = 1e-6
q = column_norm
solution = run_iteration(g_funcs, vars_symbols, initial_values, q, eps, norm=1)

new_eps: 1.1276595744680851e-06
 k         x_(1,k+1)         x_(2,k+1)         x_(3,k+1)     diff_x(1,k+1)     diff_x(2,k+1)     diff_x(3,k+1)        total_diff
 0 0.866666666666667 1.100000000000000 0.880000000000000 0.866666666666667 1.100000000000000 0.880000000000000 2.846666666666667
 1 1.020666666666667 1.112888888888889 0.933240000000000 0.154000000000000 0.012888888888889 0.053240000000000 0.220128888888889
 2 1.032962554732510 1.089147955555556 0.935133480608834 0.012295888065844 0.023740933333333 0.001893480608834 0.037930302008011
 3 1.036952710222966 1.086812184112932 0.931679777252873 0.003990155490456 0.002335771442624 0.003453703355961 0.009779630289040
 4 1.036370559031678 1.085640885401412 0.931347994629397 0.000582151191288 0.001171298711520 0.000331782623477 0.002085232526285
 5 1.036451723097495 1.085728405900176 0.931182154446110 0.000081164065817 0.000087520498764 0.000165840183287 0.000334524747868
 6 1.036394829760394 1.085694998013434 0.931194533812644 0.000056

Câu 20c:

In [12]:
def g1(x, y, z):
    return 1-sp.cos(x*y*z)

def g2(x, y, z):
    return 1-(1-x)**(1/4)-0.05*z**2+0.15*z

def g3(x, y, z):
    return x**2+0.1*y**2-0.01*y+1

g_funcs = [g1, g2, g3]
bounds = [(-0.1, 0.1), (-0.1, 0.3), (0.5, 1.1)]
check_GD(g_funcs, bounds)

--- CHI TIẾT MIỀN GIÁ TRỊ G(D) ---
g1(D): [0.000000, 0.000544] | Giới hạn D1: [-0.1, 0.1]
g2(D): [0.038386, 0.130496] | Giới hạn D2: [-0.1, 0.3]
g3(D): [0.999750, 1.016000] | Giới hạn D3: [0.5, 1.1]


In [13]:
vars_symbols = sp.symbols('x y z')
row_norm, column_norm = calc_norms(g_funcs, bounds, vars_symbols)


--- CHI TIẾT TÍNH CHUẨN HÀNG ---
Max Hàng 1 (q1): 0.0155
Max Hàng 2 (q2): 0.3706
Max Hàng 3 (q3): 0.2500
=> Chuẩn hàng q = max(0.0155, 0.3706, 0.2500) = 0.3706

--- CHI TIẾT TÍNH CHUẨN CỘT ---
Max Cột 1 (c1): 0.4814
Max Cột 2 (c2): 0.0536
Max Cột 3 (c3): 0.1004
=> Chuẩn cột q = max(0.4814, 0.0536, 0.1004) = 0.4814


In [14]:
initial_values = [0, 0, 0]
eps = 1e-6
q = row_norm
solution = run_iteration(g_funcs, vars_symbols, initial_values, q, eps, norm=-1)

new_eps: 1.6986430957535528e-06
 k  x_(1,k+1)  x_(2,k+1)  x_(3,k+1)  diff_x(1,k+1)  diff_x(2,k+1)  diff_x(3,k+1)  total_diff
 0        0.0        0.0        1.0            0.0            0.0            1.0         1.0
 1        0.0        0.1        1.0            0.0            0.1            0.0         0.1
 2        0.0        0.1        1.0            0.0            0.0            0.0         0.0
Approximate solution: [0.  0.1 1. ]


Câu 20d:

In [17]:
def g1(x, y, z):
    return ((1/3)*sp.cos(y*z))+(1/6)

def g2(x, y, z):
    return (-1/9)*sp.sqrt(x**2+sp.sin(z)+1.06)-0.1

def g3(x, y, z):
    return -1/20 * sp.exp(-x*y) - (10*sp.pi-3)/60

g_funcs = [g1, g2, g3]
bounds = [(-1.0, 1.0), (-1.0, 1.0), (-1.0, 1.0)]
check_GD(g_funcs, bounds)

--- CHI TIẾT MIỀN GIÁ TRỊ G(D) ---
g1(D): [0.346767, 0.500000] | Giới hạn D1: [-1.0, 1.0]
g2(D): [-0.289263, -0.151941] | Giới hạn D2: [-1.0, 1.0]
g3(D): [-0.609513, -0.491993] | Giới hạn D3: [-1.0, 1.0]


In [18]:
vars_symbols = sp.symbols('x y z')
row_norm, column_norm = calc_norms(g_funcs, bounds, vars_symbols)


--- CHI TIẾT TÍNH CHUẨN HÀNG ---
Max Hàng 1 (q1): 0.5610
Max Hàng 2 (q2): 0.1293
Max Hàng 3 (q3): 0.2718
=> Chuẩn hàng q = max(0.5610, 0.1293, 0.2718) = 0.5610

--- CHI TIẾT TÍNH CHUẨN CỘT ---
Max Cột 1 (c1): 0.2366
Max Cột 2 (c2): 0.4164
Max Cột 3 (c3): 0.3447
=> Chuẩn cột q = max(0.2366, 0.4164, 0.3447) = 0.4164


In [19]:
initial_values = [0, 0, 0]
eps = 1e-6
q = column_norm
solution = run_iteration(g_funcs, vars_symbols, initial_values, q, eps, norm=1)

new_eps: 1.4015114938959177e-06
 k         x_(1,k+1)          x_(2,k+1)          x_(3,k+1)     diff_x(1,k+1)     diff_x(2,k+1)     diff_x(3,k+1)        total_diff
 0 0.500000000000000 -0.214395890455411 -0.523598775598299 0.500000000000000 0.214395890455411 0.523598775598299 1.237994666053710
 1 0.497901916407003 -0.200000000000000 -0.529256504413742 0.002098083592997 0.014395890455411 0.005657728815443 0.022151702863851
 2 0.498134326654338 -0.199567869406636 -0.528834138956606 0.000232410247334 0.000432130593364 0.000422365457135 0.001086906297834
 3 0.498145333571970 -0.199604819326524 -0.528824817282544 0.000011006917632 0.000036949919888 0.000009321674063 0.000057278511582
 4 0.498144712711300 -0.199605997697567 -0.528825955119565 0.000000620860670 0.000001178371044 0.000001137837021 0.000002937068735
 5 0.498144682849644 -0.199605898481536 -0.528825980693760 0.000000029861656 0.000000099216031 0.000000025574195 0.000000154651883
Approximate solution: [ 0.49814468 -0.1996059  -0.5

## 2. Sai số tương đối:

In [15]:
def fixed_point_recursion_relative (initial_values, q, eta, *funcs, norm):
    n = len(initial_values)
    values = np.array(initial_values)
    results = [[0] + initial_values]
    i=0

    new_eta = eta * (1-q)/ q
    print(f"new_eta: {new_eta}")

    while True:
        new_values = np.array([funcs[j](*values) for j in range(n)])
        
        # Calculate the differences
        diffs = np.abs(new_values - values)
        itself = np.abs(new_values)
        if (norm == 1): 
            total_diff = np.sum(diffs)
            total_itself = np.sum(itself)
        elif (norm == -1): 
            total_diff = np.max(diffs)
            total_itself = np.max(itself)
       
        if (i!=0):
            relative_diff = total_diff / total_itself
        else:
            relative_diff = None
        
        # Append the iteration number, new values, differences, and total_diff to the results list
        results.append([i+1] + new_values.tolist() + diffs.tolist() + [total_diff] + [relative_diff])
        values = new_values
        
        # Check for convergence
        if i!=0 and relative_diff < new_eta:
            break    

        i += 1  # Increment the iteration counter 
    
    columns = ['Iteration'] + [f'x{j+1}' for j in range(n)] + [f'diff_x{j+1}' for j in range(n)] + ['total_diff'] + ['relative_diff']
    df = pd.DataFrame(results, columns=columns)
    print(df.to_string(index=False))  # Print the DataFrame without the index
    
    return values

In [16]:
def g1(x, y):
    return 1/3 * np.sin(x*y)

def g2(x, y):
    return 1/4 * np.cos(x**2+y**2)

# Store g functions
def store_g_funcs(*g_funcs):
    return g_funcs

g_funcs = store_g_funcs(g1, g2)

# Initial guess
initial_values = [1, 1]
eta = 1 * 1e-4
q = 1/3 * np.cos(1) + 1/2 * np.sin(2) #column-norm

# Perform fixed-point iteration
solution = fixed_point_recursion_relative (initial_values, q, eta, *g_funcs, norm=1)
print("Approximate solution:", solution)

new_eta: 5.75424680604918e-05
 Iteration                 x1                 x2           diff_x1           diff_x2        total_diff     relative_diff
         0  1.000000000000000  1.000000000000000               NaN               NaN               NaN               NaN
         1  0.280490328269299 -0.104036709136786 0.719509671730701 1.104036709136786 1.823546380867487               NaN
         2 -0.009725716443514  0.248999421334286 0.290216044712812 0.353036130471071 0.643252175183884 2.486237637010470
         3 -0.000807231799814  0.249518176542058 0.008918484643700 0.000518755207772 0.009437239851472 0.037699887973752
         4 -0.000067139668458  0.249515618482908 0.000740092131355 0.000002558059150 0.000742650190505 0.002975566886133
         5 -0.000005584131966  0.249515648405069 0.000061555536492 0.000000029922161 0.000061585458653 0.000246814501623
         6 -0.000000464442769  0.249515648242439 0.000005119689197 0.000000000162630 0.000005119851827 0.000020519123083
Ap